# 03 · Filled-in sentence

The pipeline embeds the black card (`… ________.`) and the white card separately, then combines the two vectors.
Here each hand card goes into the blank and the finished sentence is embedded, so the encoder reads the joke as
one text.

The first run encodes every (black card, hand card) pair of the snapshot, about 40 000 sentences: expect minutes on
idle CPU. They're cached in `ml-data/embeddings-experiments/`, and an interrupted run resumes.

In [ ]:
import os
import sys
from functools import partial
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "ml").is_dir() else Path.cwd().parent  # papermill: repo root; VS Code: notebooks/
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")

import torch
from torch import nn

from ml import experiments as ex
from ml import lower_priority
from ml.train import TrainConfig

print("CPU priority:", lower_priority())  # idle CPU only: the game comes first

In [ ]:
snapshot = None
embedding = "e5"
hidden = 64             # the MLP variant's hidden layer
center = True
lr = 1e-2
weight_decay = 1e-3
max_epochs = 600
patience = 25
log_mlflow = True

In [ ]:
data = ex.load(snapshot)
print(data.snapshot.name, data.sizes())
e = data.splits["train"][0]
for card in e.hand[:3]:
    print(ex.fill_blank(data.texts[e.black], data.texts[card]))

In [ ]:
inputs, targets = data.filled_tensors(embedding)
if center:
    inputs = ex.center(inputs)
dim = inputs["train"][0].shape[-1]
train_ex = data.splits["train"]
log_pop = {"train": ex.popularity_log_rates(train_ex, train_ex, leave_one_out=True),
           "val": ex.popularity_log_rates(train_ex, data.splits["val"]),
           "test": ex.popularity_log_rates(train_ex, data.splits["test"])}
inputs = {s: (inputs[s][0], torch.tensor(log_pop[s], dtype=torch.float32)) for s in inputs}

In [ ]:
class Filled(nn.Module):
    """Scores each filled-in sentence (n, hand, dim); optionally plus a·log(popularity rate)."""

    def __init__(self, dim, hidden=0, popularity=False, dropout=0.2):
        super().__init__()
        self.head = (nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
                     if hidden else nn.Linear(dim, 1))
        self.scale = nn.Parameter(torch.ones(())) if popularity else None

    def forward(self, filled, log_pop):
        score = self.head(filled).squeeze(-1)
        if self.scale is not None:
            score = score + self.scale * log_pop
        return score

In [ ]:
linear = TrainConfig(lr=lr, weight_decay=weight_decay, max_epochs=max_epochs, patience=patience)
mlp = TrainConfig(lr=3e-3, weight_decay=1e-4, hidden=hidden, dropout=0.2, max_epochs=max_epochs, patience=patience)
results = ex.baselines(data)
for name, config, popularity in (("filled linear", linear, False), ("filled mlp", mlp, False),
                                 ("filled linear + popularity", linear, True)):
    results.append(ex.fit(f"{name} {embedding}", partial(Filled, dim, config.hidden, popularity), inputs, targets,
                          config, {"model_type": "filled", "embedding": embedding, "popularity": popularity,
                                   "center": center}))
ex.table(results)

In [ ]:
if log_mlflow:
    print("MLflow parent run:", ex.log("03_filled_sentence", data, results, {"embedding": embedding, "center": center}))